# SiteSight M9: RGB-D visual odometry on TUM fr1/desk

Self-contained (code embedded, no GitHub token). CPU runtime is enough. Downloads the public TUM
RGB-D sequence `freiburg1_desk` (about 0.36 GB), runs frame-to-frame and keyframe RGB-D odometry
(ORB + depth + PnP/RANSAC, TUM's recommended ROS-default intrinsics, no undistortion) and scores both
against the motion-capture ground truth: ATE (own Umeyama alignment) and RPE, plus the "camera never
moved" baseline. Our ATE is cross-checked against `evo_ape` on the very same frames. The ground truth
is used only for scoring.

In [ ]:
import base64, io, os, zipfile
B64 = '''
...
'''
zipfile.ZipFile(io.BytesIO(base64.b64decode("".join(B64.split())))).extractall("/content/sitesight")
print(sorted(os.listdir("/content/sitesight")))

['README.md', 'configs', 'pyproject.toml', 'reports', 'scripts', 'sitesight']


In [ ]:
%cd /content/sitesight
%pip -q uninstall -y opencv-python opencv-contrib-python opencv-python-headless
%pip -q install -e . "opencv-python-headless>=4.9,<5" evo
import cv2, evo, os
print("cv2", cv2.__version__, "| evo", evo.__version__, "| cpus", os.cpu_count())

/content/sitesight
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sitesight (pyproject.toml) ... done
cv2 4.14.0 | evo v1.38.1 | cpus 2


In [ ]:
!curl -sSL -o /content/desk.tgz https://cvg.cit.tum.de/rgbd/dataset/freiburg1/rgbd_dataset_freiburg1_desk.tgz
!ls -la /content/desk.tgz && sha256sum /content/desk.tgz
!tar xzf /content/desk.tgz -C /content
ROOT = "/content/rgbd_dataset_freiburg1_desk"
!ls {ROOT} && ls {ROOT}/rgb | wc -l && ls {ROOT}/depth | wc -l && wc -l {ROOT}/groundtruth.txt

-rw-r--r-- 1 root root 344011403 Nov 23 17:54 /content/desk.tgz
e983d6830916e66dc4a46a71368046b149b283de87769690e7aa4e0b9483530c  /content/desk.tgz
accelerometer.txt  depth  depth.txt  groundtruth.txt  rgb  rgb.txt
613
595
2338 /content/rgbd_dataset_freiburg1_desk/groundtruth.txt


In [ ]:
!python -m sitesight.geometry.run --root {ROOT} --out-dir /content/m9

keyframe {"n_frames": 573, "n_with_ground_truth": 573, "n_tracking_failures": 0, "n_keyframes": 62, "ate_rmse_m": 0.09053093302205269, "ate_median_m": 0.07843902122304436, "ate_max_m": 0.2247610148113544, "rpe_1frame_trans_rmse_m": 0.014637217086730873, "rpe_1frame_rot_rmse_deg": 0.836503599515872, "rpe_30frames_trans_rmse_m": 0.04271637907375585, "rpe_30frames_rot_rmse_deg": 2.362597087396426, "stationary_baseline_ate_rmse_m": 0.8714043201981037, "seconds_per_frame_mean": 0.07498635983071687, "seconds_per_frame_p95": 0.11349099059996204, "config": {"mode": "keyframe", "n_features": 1500, "ratio": 0.75, "min_depth": 0.3, "max_depth": 5.0, "reproj_error": 3.0, "ransac_iters": 200, "min_inliers": 20, "kf_min_inliers": 60, "kf_max_translation": 0.15, "kf_max_rotation_deg": 15.0}}
frame2frame {"n_frames": 573, "n_with_ground_truth": 573, "n_tracking_failures": 0, "n_keyframes": 573, "ate_rmse_m": 0.08928315769338897, "ate_median_m": 0.08401408467402018, "ate_max_m": 0.1786802183192495, "rp

In [ ]:
# Cross-check our ATE against evo (same frames, SE(3) alignment, no scale).
import json, re, subprocess
m = json.load(open("/content/m9/metrics.json"))
for mode in m["modes"]:
    out = subprocess.run(
        ["evo_ape", "tum", "/content/m9/gt.txt", f"/content/m9/est_{mode}.txt", "-a", "--t_max_diff", "0.01"],
        capture_output=True, text=True,
    ).stdout
    print(out)
    m["modes"][mode]["evo_ate_rmse_m"] = float(re.search(r"rmse\s+([0-9.eE+-]+)", out).group(1))
print("M9_JSON_BEGIN")
print(json.dumps(m, separators=(",", ":"), sort_keys=True))
print("M9_JSON_END")

Initialized new /root/.evo/settings.json
APE w.r.t. translation part (m)
(with SE(3) Umeyama alignment)

       max	0.224761
      mean	0.081184
    median	0.078439
       min	0.020816
      rmse	0.090531
       sse	4.696222
       std	0.040063


APE w.r.t. translation part (m)
(with SE(3) Umeyama alignment)

       max	0.178680
      mean	0.082534
    median	0.084014
       min	0.008099
      rmse	0.089283
       sse	4.567659
       std	0.034053


M9_JSON_BEGIN
{"intrinsics":{"cx":319.5,"cy":239.5,"fx":525.0,"fy":525.0},"modes":{"frame2frame":{"ate_max_m":0.1786802183192495,"ate_median_m":0.08401408467402018,"ate_rmse_m":0.08928315769338897,"config":{"kf_max_rotation_deg":15.0,"kf_max_translation":0.15,"kf_min_inliers":60,"max_depth":5.0,"min_depth":0.3,"min_inliers":20,"mode":"frame2frame","n_features":1500,"ransac_iters":200,"ratio":0.75,"reproj_error":3.0},"evo_ate_rmse_m":0.089283,"n_frames":573,"n_keyframes":573,"n_tracking_failures":0,"n_with_ground_truth":573,"rpe_1frame_rot_rm